# 09 — Score Fusion + Re-ranking Pipeline

Takes the raw candidate pool from multimodal retrieval and produces a **final ranked list** of products.

## Why this stage exists

The retrieval stage (notebook 08) returns raw FAISS inner-product scores for text and image independently. Those scores:
- Come from the same 512-dim CLIP space but can have different distributions per query.
- Cannot be directly added without normalization — a raw text score of 0.85 is not directly comparable to a raw image score of 0.85 if the distributions differ.
- Candidates retrieved by only one modality have a missing score for the other.

## Architecture

```
Candidate Pool
      ↓
Score Normalization   (min-max per modality, per query)
      ↓
Missing Score Policy  (fill NaN with 0.0 — weakest signal)
      ↓
Score Fusion          (configurable weighted sum)
      ↓
Sort by final_score descending
      ↓
Top-K Results
```

## Why NOT training a re-ranker yet

A learned re-ranker (cross-encoder, LTR) requires labeled relevance data which we don't have yet. Deterministic score fusion gives a strong, interpretable baseline. Training comes later.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import faiss
import torch
from pathlib import Path
from PIL import Image
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device : {DEVICE}")
print(f"faiss  : {faiss.__version__}")

device : cpu
faiss  : 1.15.0


## 2. Paths

In [2]:
NOTEBOOK_DIR = Path(".").resolve()          # notebooks/
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED    = PROJECT_ROOT / "data" / "processed"
FAISS_DIR    = PROCESSED / "faiss"

PRODUCTS_CSV      = PROCESSED / "products_ml_ready.csv"
TEXT_FAISS_PATH   = FAISS_DIR  / "text_index.faiss"
IMAGE_FAISS_PATH  = FAISS_DIR  / "image_index.faiss"
TEXT_MAPPING_CSV  = FAISS_DIR  / "text_index_mapping.csv"
IMAGE_MAPPING_CSV = FAISS_DIR  / "image_index_mapping.csv"

for p in [PRODUCTS_CSV, TEXT_FAISS_PATH, IMAGE_FAISS_PATH,
          TEXT_MAPPING_CSV, IMAGE_MAPPING_CSV]:
    assert p.exists(), f"Missing file: {p}"
    print(f"  OK  {p.relative_to(PROJECT_ROOT)}")

  OK  data\processed\products_ml_ready.csv
  OK  data\processed\faiss\text_index.faiss
  OK  data\processed\faiss\image_index.faiss
  OK  data\processed\faiss\text_index_mapping.csv
  OK  data\processed\faiss\image_index_mapping.csv


## 3. Load Indexes, Mappings, and Product Metadata

In [3]:
text_index  = faiss.read_index(str(TEXT_FAISS_PATH))
image_index = faiss.read_index(str(IMAGE_FAISS_PATH))

text_mapping_df  = pd.read_csv(TEXT_MAPPING_CSV)
image_mapping_df = pd.read_csv(IMAGE_MAPPING_CSV)

text_faiss_to_pid  = dict(zip(text_mapping_df["faiss_index"],  text_mapping_df["pid"]))
image_faiss_to_pid = dict(zip(image_mapping_df["faiss_index"], image_mapping_df["pid"]))

products_df     = pd.read_csv(PRODUCTS_CSV)
products_by_pid = products_df.set_index("pid")

N_PRODUCTS = len(products_df)
FAISS_DIM  = text_index.d

# Sanity checks
assert text_index.d  == 512
assert image_index.d == 512
assert text_index.ntotal  == N_PRODUCTS
assert image_index.ntotal == N_PRODUCTS
assert products_df["pid"].duplicated().sum() == 0

print(f"Products          : {N_PRODUCTS}")
print(f"FAISS dim         : {FAISS_DIM}")
print(f"text_index ntotal : {text_index.ntotal}")
print(f"image_index ntotal: {image_index.ntotal}")
print("All sanity checks passed.")

Products          : 4681
FAISS dim         : 512
text_index ntotal : 4681
image_index ntotal: 4681
All sanity checks passed.


## 4. Load CLIP Model

In [4]:
MODEL_NAME = "openai/clip-vit-base-patch32"

print(f"Loading {MODEL_NAME} ...")
clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip_tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model.eval()
print(f"Model ready on : {DEVICE}")

Loading openai/clip-vit-base-patch32 ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model ready on : cpu


## 5. Retrieval Layer (from Notebook 08)

Replicated here so this notebook is self-contained. No logic changes.

In [5]:
def _l2_normalize(vec: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(vec, axis=1, keepdims=True)
    return vec / np.clip(norm, 1e-10, None)


def encode_text(query: str) -> np.ndarray:
    if not query or not query.strip():
        raise ValueError("Text query must be a non-empty string.")
    tokens = clip_tokenizer([query.strip()], return_tensors="pt",
                             padding=True, truncation=True, max_length=77)
    tokens = {k: v.to(DEVICE) for k, v in tokens.items()}
    with torch.no_grad():
        out = clip_model.text_model(**tokens)
        emb = clip_model.text_projection(out.pooler_output)
    return _l2_normalize(emb.cpu().float().numpy())


def encode_image(image_path: str) -> np.ndarray:
    if not image_path:
        raise ValueError("image_path must be provided.")
    path = Path(image_path)
    if not path.is_absolute():
        path = (NOTEBOOK_DIR / path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")
    try:
        img = Image.open(path).convert("RGB")
    except Exception as e:
        raise IOError(f"Cannot open image {path}: {e}")
    inputs = clip_processor(images=[img], return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE)
    with torch.no_grad():
        vis = clip_model.vision_model(pixel_values=pixel_values)
        emb = clip_model.visual_projection(vis.pooler_output)
    return _l2_normalize(emb.cpu().float().numpy())


def _faiss_search(index, faiss_to_pid, query_vec, top_k) -> list:
    scores, indices = index.search(query_vec.astype(np.float32), top_k)
    results = []
    for fidx, score in zip(indices[0], scores[0]):
        if fidx == -1:
            continue
        pid = faiss_to_pid.get(int(fidx))
        if pid is None or pid not in products_by_pid.index:
            continue
        results.append({"faiss_index": int(fidx), "pid": pid, "score": float(score)})
    return results


def _raw_candidates(text_query=None, image_path=None, top_k=20) -> pd.DataFrame:
    """
    Return raw candidate pool with text_score / image_score columns (NaN if not available).
    """
    has_text  = text_query is not None and str(text_query).strip() != ""
    has_image = image_path is not None and str(image_path).strip() != ""

    if not has_text and not has_image:
        raise ValueError("Provide at least one of: text_query or image_path.")

    text_results  = []
    image_results = []

    if has_text:
        qvec = encode_text(text_query)
        text_results = _faiss_search(text_index, text_faiss_to_pid, qvec, top_k)

    if has_image:
        qvec = encode_image(image_path)
        image_results = _faiss_search(image_index, image_faiss_to_pid, qvec, top_k)

    t_df = pd.DataFrame(text_results).rename(columns={"score": "text_score"})
    i_df = pd.DataFrame(image_results).rename(columns={"score": "image_score"})

    if not t_df.empty and not i_df.empty:
        merged = t_df[["pid", "text_score"]].merge(
                 i_df[["pid", "image_score"]], on="pid", how="outer")
    elif not t_df.empty:
        merged = t_df[["pid", "text_score"]].copy()
        merged["image_score"] = np.nan
    else:
        merged = i_df[["pid", "image_score"]].copy()
        merged["text_score"] = np.nan

    def _tag(row):
        ht = pd.notna(row["text_score"])
        hi = pd.notna(row["image_score"])
        return "both" if (ht and hi) else ("text" if ht else "image")

    merged["retrieved_by"] = merged.apply(_tag, axis=1)
    return merged.reset_index(drop=True)


print("Retrieval layer loaded.")

Retrieval layer loaded.


## 6. Inspect Raw Score Distributions

Before normalizing, we look at what the raw FAISS scores actually look like.
Both scores come from the same 512-dim CLIP space, but their per-query ranges can differ.

In [6]:
# Use a real multimodal query for inspection — image selected programmatically
_inspect_text  = "women's leggings"
_inspect_image = products_df[
    products_df["main_category"].str.lower().str.contains("clothing", na=False)
].iloc[0]["image_path"]

raw = _raw_candidates(text_query=_inspect_text, image_path=_inspect_image, top_k=20)

print("Raw score statistics (before normalization)")
print(f"  Candidates total : {len(raw)}")
print(f"  retrieved by both : {(raw['retrieved_by']=='both').sum()}")
print()
for col in ["text_score", "image_score"]:
    vals = raw[col].dropna()
    print(f"  {col}:")
    print(f"    count  : {len(vals)}")
    print(f"    min    : {vals.min():.4f}")
    print(f"    max    : {vals.max():.4f}")
    print(f"    mean   : {vals.mean():.4f}")
    print(f"    std    : {vals.std():.4f}")
    print()

Raw score statistics (before normalization)
  Candidates total : 32
  retrieved by both : 8

  text_score:
    count  : 20
    min    : 0.6930
    max    : 0.7784
    mean   : 0.7232
    std    : 0.0269

  image_score:
    count  : 20
    min    : 0.8428
    max    : 1.0000
    mean   : 0.8736
    std    : 0.0360



## 7. Score Normalization

### Why normalize?

Raw FAISS inner-product scores from text and image retrievals can have different distributions even within the same CLIP space — the top text score for a query might be 0.85 while the top image score is 0.93. Directly weighting these without normalization would bias the fusion toward whichever modality happens to produce higher absolute values.

**Method: min-max normalization per modality, per query.**

Each score column is independently scaled to `[0, 1]` based on the min and max values within the current candidate pool. This makes both modalities equally scaled before fusion.

### Handling missing scores

A candidate retrieved only by text has `image_score = NaN`. After normalization, its `normalized_image_score` is filled with `0.0` — the weakest possible signal. This is conservative and explicit: the missing modality contributes nothing rather than a made-up value. The same applies in reverse for image-only candidates.

In [7]:
def normalize_scores(candidates: pd.DataFrame) -> pd.DataFrame:
    """
    Min-max normalize text_score and image_score independently.

    - Normalization is per-query (applied to the current candidate set only).
    - Columns normalized: text_score → normalized_text_score
                          image_score → normalized_image_score
    - NaN values (missing modality) are normalized to 0.0.
    - If all values in a column are identical, normalized score = 1.0.

    Parameters
    ----------
    candidates : pd.DataFrame with text_score and image_score columns.

    Returns
    -------
    pd.DataFrame with added normalized_text_score and normalized_image_score columns.
    """
    result = candidates.copy()

    for raw_col, norm_col in [("text_score", "normalized_text_score"),
                               ("image_score", "normalized_image_score")]:
        vals = result[raw_col].dropna()

        if vals.empty:
            # No scores for this modality at all — set everything to 0
            result[norm_col] = 0.0
            continue

        v_min = float(vals.min())
        v_max = float(vals.max())
        spread = v_max - v_min

        if spread < 1e-10:
            # All values identical — normalize to 1.0 for present, 0.0 for absent
            result[norm_col] = result[raw_col].apply(
                lambda x: 1.0 if pd.notna(x) else 0.0
            )
        else:
            # Standard min-max
            result[norm_col] = result[raw_col].apply(
                lambda x: float((x - v_min) / spread) if pd.notna(x) else 0.0
            )

    return result


# Test on the inspection candidates
raw_norm = normalize_scores(raw)
print("After normalization:")
for col in ["normalized_text_score", "normalized_image_score"]:
    vals = raw_norm[col]
    print(f"  {col}: min={vals.min():.4f}  max={vals.max():.4f}  NaN={vals.isna().sum()}")

After normalization:
  normalized_text_score: min=0.0000  max=1.0000  NaN=0
  normalized_image_score: min=0.0000  max=1.0000  NaN=0


## 8. Score Fusion

### How fusion works

```
final_score = text_weight * normalized_text_score
            + image_weight * normalized_image_score
```

Weights are **configurable per call** — defaults are `text_weight=0.5, image_weight=0.5`.

No constraint is enforced that weights must sum to 1, giving callers full flexibility. For text-only or image-only queries, the absent modality contributes 0 regardless of its weight, so the effective ranking reduces to a rescaled version of the single available score.

### Design choices

- Equal default weights (0.5 / 0.5) treat both modalities symmetrically.
- Text-only: `final_score = text_weight * normalized_text_score`.
- Image-only: `final_score = image_weight * normalized_image_score`.
- Multimodal: both terms contribute.

In [8]:
def fuse_scores(candidates: pd.DataFrame,
                text_weight: float = 0.5,
                image_weight: float = 0.5) -> pd.DataFrame:
    """
    Compute final_score = text_weight * normalized_text_score
                        + image_weight * normalized_image_score.

    Requires normalized_text_score and normalized_image_score columns
    (produced by normalize_scores).

    Parameters
    ----------
    candidates   : pd.DataFrame with normalized score columns
    text_weight  : float — weight for normalized text score (default 0.5)
    image_weight : float — weight for normalized image score (default 0.5)

    Returns
    -------
    pd.DataFrame with added final_score column.
    """
    if text_weight < 0 or image_weight < 0:
        raise ValueError("Weights must be non-negative.")
    if text_weight == 0 and image_weight == 0:
        raise ValueError("At least one weight must be > 0.")

    result = candidates.copy()
    result["final_score"] = (
        text_weight  * result["normalized_text_score"] +
        image_weight * result["normalized_image_score"]
    )
    return result


print("fuse_scores defined.")

fuse_scores defined.


## 9. Re-ranking: `rerank_candidates`

Applies normalization → fusion → sort → top-K in one step.

In [9]:
def rerank_candidates(candidates: pd.DataFrame,
                      text_weight: float = 0.5,
                      image_weight: float = 0.5,
                      top_k: int = 10) -> pd.DataFrame:
    """
    Normalize, fuse, and sort a candidate pool into a final ranking.

    Parameters
    ----------
    candidates   : raw candidate pool from _raw_candidates()
    text_weight  : fusion weight for normalized text score
    image_weight : fusion weight for normalized image score
    top_k        : number of final results to return

    Returns
    -------
    pd.DataFrame with columns:
        rank, pid, product_name, main_category, brand, image_path,
        text_score, image_score,
        normalized_text_score, normalized_image_score,
        final_score, retrieved_by
    Sorted by final_score descending.
    """
    # Normalize
    normed = normalize_scores(candidates)

    # Fuse
    fused = fuse_scores(normed, text_weight=text_weight, image_weight=image_weight)

    # Sort descending by final_score
    fused = fused.sort_values("final_score", ascending=False).reset_index(drop=True)

    # Top-K
    fused = fused.head(top_k).reset_index(drop=True)

    # Attach product metadata
    meta_cols = ["pid", "product_name", "main_category", "brand", "image_path"]
    fused = fused.merge(products_df[meta_cols], on="pid", how="left")

    # Add rank column
    fused.insert(0, "rank", range(1, len(fused) + 1))

    # Final column order
    col_order = ["rank", "pid", "product_name", "main_category", "brand",
                 "image_path", "text_score", "image_score",
                 "normalized_text_score", "normalized_image_score",
                 "final_score", "retrieved_by"]
    available_cols = [c for c in col_order if c in fused.columns]
    return fused[available_cols]


print("rerank_candidates defined.")

rerank_candidates defined.


## 10. Unified Entry Point: `rank_products`

Single function covering all three query modes.

In [10]:
def rank_products(text_query: str | None = None,
                  image_path: str | None = None,
                  top_k: int = 10,
                  retrieval_k: int = 40,
                  text_weight: float = 0.5,
                  image_weight: float = 0.5) -> pd.DataFrame:
    """
    Full pipeline: query → retrieval → candidate pool → re-ranking → top-K.

    Parameters
    ----------
    text_query   : natural-language query, or None
    image_path   : path to query image, or None
    top_k        : number of final ranked results
    retrieval_k  : number of candidates fetched per FAISS index (> top_k recommended)
    text_weight  : fusion weight for text modality
    image_weight : fusion weight for image modality

    Returns
    -------
    pd.DataFrame — final ranked product list.
    """
    candidates = _raw_candidates(text_query=text_query,
                                 image_path=image_path,
                                 top_k=retrieval_k)
    return rerank_candidates(candidates,
                             text_weight=text_weight,
                             image_weight=image_weight,
                             top_k=top_k)


print("rank_products defined.")

rank_products defined.


## 11. Execute — Text-Only Ranking

In [11]:
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", "{:.4f}".format)

text_query = "men's formal shirt"
results_text = rank_products(text_query=text_query, top_k=10, retrieval_k=40)

print(f"Text-only ranking — '{text_query}'")
print(f"Results: {len(results_text)} | retrieved_by: {results_text['retrieved_by'].unique().tolist()}")
print()
print(results_text[["rank","product_name","main_category",
                     "text_score","normalized_text_score",
                     "final_score","retrieved_by"]].to_string(index=False))

Text-only ranking — 'men's formal shirt'
Results: 10 | retrieved_by: ['text']

 rank                                                             product_name main_category  text_score  normalized_text_score  final_score retrieved_by
    1 Jorzzer Roniya Men's Solid Formal, Party, Wedding, Casual, Festive Shirt      Clothing      0.6748                 1.0000       0.5000         text
    2                                       Stylenara Men's Solid Casual Shirt      Clothing      0.6500                 0.7388       0.3694         text
    3                                   Hoffmen Men's Self Design Formal Shirt      Clothing      0.6365                 0.5968       0.2984         text
    4                                            Leaf Men's Solid Formal Shirt      Clothing      0.6343                 0.5736       0.2868         text
    5                                                  Sonata 77036SM02J Watch       Watches      0.6272                 0.4988       0.2494         te

## 12. Execute — Image-Only Ranking

In [12]:
# Select a query image programmatically from the Footwear category
fw_row        = products_df[products_df["main_category"].str.lower()
                             .str.contains("footwear", na=False)].iloc[0]
query_img     = fw_row["image_path"]
query_pid     = fw_row["pid"]
query_product = fw_row["product_name"]

results_image = rank_products(image_path=query_img, top_k=10, retrieval_k=40)

print(f"Image-only ranking")
print(f"Query image   : {query_img}")
print(f"Query product : {query_product} ({fw_row['main_category']})")
print(f"Results       : {len(results_image)} | retrieved_by: {results_image['retrieved_by'].unique().tolist()}")
print()
print(results_image[["rank","product_name","main_category",
                      "image_score","normalized_image_score",
                      "final_score","retrieved_by"]].to_string(index=False))
print(f"\nQuery product at rank 1: {results_image.iloc[0]['pid'] == query_pid}  (expected True)")

Image-only ranking
Query image   : ../data/images/SNDEDAPKZGGEGYHV.jpg
Query product : S.m.a.R.T FEET Women Wedges (Footwear)
Results       : 10 | retrieved_by: ['image']

 rank                                          product_name main_category  image_score  normalized_image_score  final_score retrieved_by
    1                           S.m.a.R.T FEET Women Wedges      Footwear       1.0000                  1.0000       0.5000        image
    2                                femitaly Women Bellies      Footwear       0.8945                  0.2761       0.1380        image
    3 Lord's Antique Gold Women's Peeptoe Heels Women Heels      Footwear       0.8840                  0.2041       0.1020        image
    4                                        Bonzer Bellies      Footwear       0.8827                  0.1952       0.0976        image
    5                                     Wellworth Loafers      Footwear       0.8805                  0.1797       0.0898        image
    6 

## 13. Execute — True Multimodal Ranking (Text + Image)

In [13]:
mm_text  = "running shoes for men"
# Pick query image from Footwear — programmatic, not hardcoded
mm_img   = products_df[products_df["main_category"].str.lower()
                        .str.contains("footwear", na=False)].iloc[1]["image_path"]

results_mm = rank_products(
    text_query   = mm_text,
    image_path   = mm_img,
    top_k        = 10,
    retrieval_k  = 40,
    text_weight  = 0.5,
    image_weight = 0.5,
)

both_count  = (results_mm["retrieved_by"] == "both").sum()
text_only   = (results_mm["retrieved_by"] == "text").sum()
image_only  = (results_mm["retrieved_by"] == "image").sum()

print(f"Multimodal ranking — text: '{mm_text}' + image: {mm_img}")
print(f"Results       : {len(results_mm)}")
print(f"  by both     : {both_count}")
print(f"  text only   : {text_only}")
print(f"  image only  : {image_only}")
print()
print(results_mm[["rank","product_name","main_category",
                   "normalized_text_score","normalized_image_score",
                   "final_score","retrieved_by"]].to_string(index=False))

Multimodal ranking — text: 'running shoes for men' + image: ../data/images/SHOECFFWRJYBTPHY.jpg
Results       : 10
  by both     : 0
  text only   : 9
  image only  : 1

 rank                             product_name main_category  normalized_text_score  normalized_image_score  final_score retrieved_by
    1                           Credos Loafers      Footwear                 0.0000                  1.0000       0.5000        image
    2                      People Casual Shoes      Footwear                 1.0000                  0.0000       0.5000         text
    3   Port Sport-Shiled Training & Gym Shoes      Footwear                 0.9890                  0.0000       0.4945         text
    4                  99Moves MOV-269-9 Boots      Footwear                 0.9584                  0.0000       0.4792         text
    5       JQR JQR Sports Shoes Running Shoes      Footwear                 0.9534                  0.0000       0.4767         text
    6 Tennis Tennis Sports

## 14. Demonstrate Configurable Weights

The same query with different weight configurations to show the ranking changes.

In [14]:
configs = [
    ("Text-heavy  (0.8/0.2)", 0.8, 0.2),
    ("Balanced    (0.5/0.5)", 0.5, 0.5),
    ("Image-heavy (0.2/0.8)", 0.2, 0.8),
]

for label, tw, iw in configs:
    r = rank_products(
        text_query=mm_text, image_path=mm_img,
        top_k=5, retrieval_k=40,
        text_weight=tw, image_weight=iw,
    )
    print(f"\n--- {label} ---")
    print(r[["rank","product_name","main_category",
              "normalized_text_score","normalized_image_score",
              "final_score","retrieved_by"]].to_string(index=False))


--- Text-heavy  (0.8/0.2) ---
 rank                             product_name main_category  normalized_text_score  normalized_image_score  final_score retrieved_by
    1                      People Casual Shoes      Footwear                 1.0000                  0.0000       0.8000         text
    2   Port Sport-Shiled Training & Gym Shoes      Footwear                 0.9890                  0.0000       0.7912         text
    3                  99Moves MOV-269-9 Boots      Footwear                 0.9584                  0.0000       0.7668         text
    4       JQR JQR Sports Shoes Running Shoes      Footwear                 0.9534                  0.0000       0.7627         text
    5 Tennis Tennis Sports Shoes Running Shoes      Footwear                 0.8749                  0.0000       0.6999         text



--- Balanced    (0.5/0.5) ---
 rank                           product_name main_category  normalized_text_score  normalized_image_score  final_score retrieved_by
    1                         Credos Loafers      Footwear                 0.0000                  1.0000       0.5000        image
    2                    People Casual Shoes      Footwear                 1.0000                  0.0000       0.5000         text
    3 Port Sport-Shiled Training & Gym Shoes      Footwear                 0.9890                  0.0000       0.4945         text
    4                99Moves MOV-269-9 Boots      Footwear                 0.9584                  0.0000       0.4792         text
    5     JQR JQR Sports Shoes Running Shoes      Footwear                 0.9534                  0.0000       0.4767         text

--- Image-heavy (0.2/0.8) ---
 rank              product_name main_category  normalized_text_score  normalized_image_score  final_score retrieved_by
    1            Credos Loa

## 15. Verification

In [15]:
def verify_ranked(results: pd.DataFrame, label: str, expected_top_k: int):
    assert len(results) <= expected_top_k,      f"{label}: too many rows"
    assert results["pid"].duplicated().sum() == 0, f"{label}: dup PIDs"
    bad = [p for p in results["pid"] if p not in products_by_pid.index]
    assert len(bad) == 0, f"{label}: invalid PIDs {bad}"
    assert results["final_score"].is_monotonic_decreasing or \
           results["final_score"].iloc[0] >= results["final_score"].iloc[-1], \
           f"{label}: not sorted descending"
    assert results["final_score"].notna().all(), f"{label}: NaN final_score"
    assert np.isfinite(results["final_score"].values).all(), f"{label}: Inf final_score"
    assert results["normalized_text_score"].between(0, 1).all(),  f"{label}: norm_text out of [0,1]"
    assert results["normalized_image_score"].between(0, 1).all(), f"{label}: norm_image out of [0,1]"
    print(f"  {label}: {len(results)} results, sorted, no dup PIDs, scores in [0,1]  ✓")

print("=== Verification ===")
verify_ranked(results_text,  "Text-only",  10)
verify_ranked(results_image, "Image-only", 10)
verify_ranked(results_mm,    "Multimodal", 10)
print("\nAll verifications passed.")

=== Verification ===
  Text-only: 10 results, sorted, no dup PIDs, scores in [0,1]  ✓
  Image-only: 10 results, sorted, no dup PIDs, scores in [0,1]  ✓
  Multimodal: 10 results, sorted, no dup PIDs, scores in [0,1]  ✓

All verifications passed.


## 16. Final Report

In [16]:
print("=" * 58)
print("SCORE FUSION + RE-RANKING — FINAL REPORT")
print("=" * 58)
print(f"Products available       : {N_PRODUCTS}")
print(f"FAISS dim (text/image)   : {text_index.d} / {image_index.d}")
print()
print("Normalization method      : min-max per modality per query")
print("Fusion method             : weighted sum of normalized scores")
print("Default weights           : text=0.5, image=0.5 (configurable)")
print("Missing score handling    : NaN → 0.0 (absent modality contributes nothing)")
print()
print("Functions implemented:")
print("  normalize_scores(candidates)              → normalized score columns")
print("  fuse_scores(candidates, tw, iw)           → final_score column")
print("  rerank_candidates(candidates, tw, iw, k)  → sorted ranked DataFrame")
print("  rank_products(text, image, k, rk, tw, iw) → unified pipeline")
print()
print(f"Text-only ranking   : {len(results_text)} results  ✓")
print(f"Image-only ranking  : {len(results_image)} results  ✓")
print(f"Multimodal ranking  : {len(results_mm)} results  ✓")
print(f"  - by both         : {(results_mm['retrieved_by']=='both').sum()}")
print(f"  - text only       : {(results_mm['retrieved_by']=='text').sum()}")
print(f"  - image only      : {(results_mm['retrieved_by']=='image').sum()}")
print()
print("Verification          : PASSED (no dup PIDs, sorted, finite, in [0,1])")
print("Learned re-ranker     : NOT implemented (deterministic baseline first)")
print()
print("Status: COMPLETE")
print("Next stage: Backend API / Frontend")
print("=" * 58)

SCORE FUSION + RE-RANKING — FINAL REPORT
Products available       : 4681
FAISS dim (text/image)   : 512 / 512

Normalization method      : min-max per modality per query
Fusion method             : weighted sum of normalized scores
Default weights           : text=0.5, image=0.5 (configurable)
Missing score handling    : NaN → 0.0 (absent modality contributes nothing)

Functions implemented:
  normalize_scores(candidates)              → normalized score columns
  fuse_scores(candidates, tw, iw)           → final_score column
  rerank_candidates(candidates, tw, iw, k)  → sorted ranked DataFrame
  rank_products(text, image, k, rk, tw, iw) → unified pipeline

Text-only ranking   : 10 results  ✓
Image-only ranking  : 10 results  ✓
Multimodal ranking  : 10 results  ✓
  - by both         : 0
  - text only       : 9
  - image only      : 1

Verification          : PASSED (no dup PIDs, sorted, finite, in [0,1])
Learned re-ranker     : NOT implemented (deterministic baseline first)

Status: COM